# 用 LangChain 写一个学习搭子

选择 `langchain_env` 内核，从上往下运行一次。在聊天输入框中打字，按 Enter 发送；收到回答后可以直接追问。输入“退出”结束循环。之后单独重跑聊天格，可以接着当前内核里的历史继续聊。

In [1]:
from dotenv import dotenv_values
from langchain_core.messages import ChatMessage, HumanMessage, trim_messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

# 从本目录读取密钥和代理，缺少任意一项就停止。
config = dotenv_values(".env")
if not config.get("OPENAI_API_KEY") or not config.get("OPENAI_PROXY"):
    raise ValueError("请在本目录的 .env 中填写 OPENAI_API_KEY 和 OPENAI_PROXY。")

# 练习时可调整这两个预算，单位都是 token。
MAX_OUTPUT_TOKENS = 512    # 单次生成的硬上限，包含模型推理和最终回复。
HISTORY_TOKEN_BUDGET = 2048  # 每轮发给模型的历史问答预算，按本地估算值裁剪。

model = ChatOpenAI(
    model="gpt-6-luna",
    max_completion_tokens=MAX_OUTPUT_TOKENS,
    api_key=config["OPENAI_API_KEY"],
    openai_proxy=config["OPENAI_PROXY"],
    base_url="https://api.openai.com/v1",
    use_responses_api=True,
)

In [2]:
# 花括号标出待填的变量。模板规定每次调用时消息的角色和顺序。
topic = "Python"
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一位帮助初学者学习{topic}的中文老师。"),
    ChatMessage(role="developer", content="先用一句话回答，再给一个简短例子。"),
    MessagesPlaceholder("history"),  # 在这里插入之前的用户提问和 AI 回复。
    ("human", "{question}"),         # LangChain 的 human 对应 OpenAI 的 user。
])

# 只展开模板，在本地检查角色和内容；这里不会调用模型。
preview = prompt.invoke({"topic": topic, "history": [], "question": "什么是列表？"})
print(preview.to_string())

System: 你是一位帮助初学者学习Python的中文老师。
developer: 先用一句话回答，再给一个简短例子。
Human: 什么是列表？


In [3]:
# 管道符让模板先填入变量，再把得到的消息列表交给模型。
chain = prompt | model
history = []  # 重新运行本格会清空历史，开始一段新对话。

def chat(question):
    # 历史预算只计算以前的问答。system、developer 和当前问题仍由模板另外填入。
    recent_history = trim_messages(
        history,
        max_tokens=HISTORY_TOKEN_BUDGET,
        token_counter="approximate",  # 本地估算，实际计费以 API 返回的用量为准。
        strategy="last",              # 超过预算时优先保留最近的完整消息。
        start_on="human",             # 从用户提问开始，避免只留下孤立的回复。
    )
    reply = chain.invoke({"topic": topic, "history": recent_history, "question": question})
    # reply 是 AIMessage，对应 assistant。保留消息对象，下一轮才能区分谁问、谁答。
    history.extend([HumanMessage(content=question), reply])
    return reply.text

In [ ]:
# 只运行本格一次。每次收到回复后，循环会显示下一个输入框，chat() 会保存这一轮历史。
while True:
    question = input("你（输入「退出」结束）：").strip()
    if question == "退出":
        break
    if question:
        print("学习搭子：", chat(question))